In [27]:
# %pip install -r requirements.txt

## Import libraries

In [28]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset, load_from_disk
from collections import Counter
import pandas as pd
import numpy as np

import datasets
datasets.logging.set_verbosity_error()

## Load datasets

In [29]:
review_dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_Electronics", split="full", trust_remote_code=True)

## Data preprocessing

### Review dataset

Check dataset features

In [30]:
review_dataset

Dataset({
    features: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase'],
    num_rows: 43886944
})

Remove unwanted or uninformative columns

In [31]:
review_dataset = review_dataset.remove_columns(['asin', 'images', 'timestamp', 'helpful_vote'])

In [32]:
print(review_dataset.unique('rating'))
print(review_dataset.unique('verified_purchase'))

[3.0, 1.0, 5.0, 4.0, 2.0, 0.0]
[True, False]


We can filter the dataset to exclude reviews with 0.0 rating and unverified_purchases

In [33]:
review_dataset = review_dataset.filter(lambda x: x['verified_purchase'] == True and x['rating'] != 0, num_proc=10)

In [34]:
review_dataset = review_dataset.remove_columns(['verified_purchase'])

In [35]:
review_dataset.save_to_disk('datasets/review_dataset_attrfiltered', num_proc=10)

Saving the dataset (27/27 shards): 100%|██████████| 40546882/40546882 [03:06<00:00, 217678.02 examples/s]


### Item dataset

In [36]:
item_dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Electronics", split="full", trust_remote_code=True)

In [37]:
item_dataset

Dataset({
    features: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author'],
    num_rows: 1610012
})

In [38]:
item_dataset[10]

{'main_category': 'Computers',
 'title': 'Network Magic 4.0',
 'average_rating': 3.3,
 'rating_number': 14,
 'features': ['Connect, Manage, and Secure Your Network',
  'Share printers and files between all of your computers',
  'Connect and repair your network and internet connection',
  'View a live map of network; protect wireless network from intruders',
  'Print from any computer; monitor Internet use and Web sites visited'],
 'description': ['Product description',
  'You try to connect your computers and devices together to share a printer, files and maintain an internet connection, but find yourself wasting hours sorting through confusing dialog boxes. It is time to stop with the hassles and let Network Magic do the work for you! With Network Magic you can easily 1) connect, manage, and secure your network; 2) share printers and files between all of your computers; 3) monitor Internet use and web sites visited; 4) connect and repair your network and internet connection; and 5) Pr

Remove unwanted columns

In [39]:
item_dataset = item_dataset.remove_columns(['main_category',
                            'categories',
                            'bought_together', 
                            'store', 
                            'subtitle', 
                            'author', 
                            'videos',
                            'details'])

In [40]:
item_dataset[4]

{'title': 'Motorola Droid X Essentials Combo Pack',
 'average_rating': 3.8,
 'rating_number': 64,
 'features': ['New Droid X Essentials Combo Pack',
  'Exclusive Package Incredible Value Worth $145!!!',
  'Includes all Genuine High Quality Motorola Made Accessories'],
 'description': ['all Genuine High Quality Motorola Made Accessories, including Multimedia Station with HDMI technology, HDMI Cable and AC Wall Charger, Motorola Navigation / Music Vehicle Charging Mount Car Dock and Motorola 12v Vehicle Power Adapter Charger!'],
 'price': '14.99',
 'images': {'hi_res': [None, None, None, None, None],
  'large': ['https://m.media-amazon.com/images/I/51-DXSMlHaL._AC_.jpg',
   'https://m.media-amazon.com/images/I/31HYf51qCQL._AC_.jpg',
   'https://m.media-amazon.com/images/I/31P1+fPxUNL._AC_.jpg',
   'https://m.media-amazon.com/images/I/31nLJFuQiXL._AC_.jpg',
   'https://m.media-amazon.com/images/I/21DyMQ6Bk5L._AC_.jpg'],
  'thumb': ['https://m.media-amazon.com/images/I/51-DXSMlHaL._AC_SR38

In [41]:
item_dataset.save_to_disk('datasets/item_dataset_attrfiltered', num_proc=10)

Saving the dataset (10/10 shards): 100%|██████████| 1610012/1610012 [00:09<00:00, 163994.59 examples/s]


## Attr filtered dataset

In [42]:
filtered_review_dataset = load_from_disk('datasets/review_dataset_attrfiltered')
filtered_review_dataset

Dataset({
    features: ['rating', 'title', 'text', 'parent_asin', 'user_id'],
    num_rows: 40546882
})

In [43]:
filtered_item_dataset = load_from_disk('datasets/item_dataset_attrfiltered')
filtered_item_dataset

Dataset({
    features: ['title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'parent_asin'],
    num_rows: 1610012
})

Update rating numbers

In [44]:
asin_counts = Counter(filtered_review_dataset['parent_asin'])

filtered_item_dataset = filtered_item_dataset.map(lambda x, counts=asin_counts: {'rating_number': counts.get(x['parent_asin'], 0)}, num_proc=10)

Map (num_proc=10): 100%|██████████| 1610012/1610012 [01:37<00:00, 16570.76 examples/s] 


Filter products without price or less than 50 reviews

In [45]:
min_rating = 30
filtered_item_dataset = filtered_item_dataset.filter(lambda x, m=min_rating: x['rating_number'] >= m and x['price'] is not None, num_proc=10)

Filter (num_proc=10): 100%|██████████| 1610012/1610012 [00:29<00:00, 55286.04 examples/s]


Compare with item dataset to get rid of reviews of products that aren't in dataset

In [46]:
print(len(filtered_item_dataset.unique('parent_asin')))
print(len(filtered_review_dataset.unique('parent_asin')))

Flattening the indices: 100%|██████████| 166942/166942 [00:33<00:00, 4943.34 examples/s]


166942
1535611


In [47]:
missing_asins = set(filtered_review_dataset.unique('parent_asin')) - set(filtered_item_dataset.unique('parent_asin'))
print(len(missing_asins))

1368669


In [48]:
filtered_review_dataset = filtered_review_dataset.filter(lambda x, S=missing_asins: x['parent_asin'] not in S, num_proc=10)

Filter (num_proc=10): 100%|██████████| 40546882/40546882 [02:03<00:00, 327932.30 examples/s] 


In [49]:
filtered_review_dataset.save_to_disk('datasets/review_dataset_asinfiltered', num_proc=10)

Saving the dataset (21/21 shards): 100%|██████████| 34105969/34105969 [02:11<00:00, 258614.19 examples/s]


In [50]:
filtered_item_dataset.save_to_disk('datasets/item_dataset_asinfiltered', num_proc=10)

Saving the dataset (10/10 shards): 100%|██████████| 166942/166942 [00:07<00:00, 21420.07 examples/s]


## Asin filtered dataset

In [51]:
asin_review_dataset = load_from_disk('datasets/review_dataset_asinfiltered')
asin_review_dataset

Dataset({
    features: ['rating', 'title', 'text', 'parent_asin', 'user_id'],
    num_rows: 34105969
})

In [52]:
asin_item_dataset = load_from_disk('datasets/item_dataset_asinfiltered')
asin_item_dataset

Dataset({
    features: ['title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'parent_asin'],
    num_rows: 166942
})

Remove reviews made by users with 3 reviews or less

In [53]:
#Remove reviews made by users with 5 reviews or less
min_reviews_per_user = 5
user_counts = Counter(asin_review_dataset['user_id'])

valid_user_ids = {user_id for user_id, count in user_counts.items() if count > min_reviews_per_user}

In [54]:
asin_review_dataset = asin_review_dataset.filter(lambda x, S=valid_user_ids: x['user_id'] in S, num_proc=10)

Filter (num_proc=10): 100%|██████████| 34105969/34105969 [01:28<00:00, 387109.87 examples/s] 


In [55]:
asin_review_dataset

Dataset({
    features: ['rating', 'title', 'text', 'parent_asin', 'user_id'],
    num_rows: 9928825
})

Recount ratings and averages

In [56]:
pd_df = asin_review_dataset.select_columns(['parent_asin', 'rating']).to_pandas()
stats = pd_df.groupby('parent_asin')['rating'].agg(['mean', 'count'])
stats['mean'] = stats['mean'].round(2)

mean_stats = stats['mean'].to_dict()
count_stats = stats['count'].to_dict()


Check for missing products

In [57]:
missing_asins = set(asin_item_dataset.unique('parent_asin')) - set(asin_review_dataset.unique('parent_asin'))

asin_item_dataset = asin_item_dataset.filter(lambda x, S=missing_asins: x['parent_asin'] not in S, num_proc=10)

Filter (num_proc=10): 100%|██████████| 166942/166942 [00:12<00:00, 13839.93 examples/s]


In [58]:
def update_numbers(x, mean_stats=mean_stats, count_stats=count_stats):
    idx = x['parent_asin']
    x['rating_number'] = count_stats.get(idx,0)
    x['average_rating'] = mean_stats.get(idx,0)
    return x

asin_item_dataset = asin_item_dataset.map(update_numbers, num_proc=10)

Map (num_proc=10): 100%|██████████| 166703/166703 [00:22<00:00, 7427.01 examples/s] 


In [59]:
asin_item_dataset.save_to_disk('datasets/item_dataset_final', num_proc=10)
asin_review_dataset.save_to_disk('datasets/review_dataset_final', num_proc=10)

Saving the dataset (10/10 shards): 100%|██████████| 9928825/9928825 [03:21<00:00, 49223.68 examples/s]


## Final filtered dataset

In [60]:
final_review_dataset = load_from_disk('datasets/review_dataset_final')
final_item_dataset = load_from_disk('datasets/item_dataset_final')

In [61]:
final_item_dataset

Dataset({
    features: ['title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'parent_asin'],
    num_rows: 166703
})

Select the top 2000 with more ratings

In [62]:
final_item_dataset=final_item_dataset.sort('rating_number', reverse=True).select(range(2000))

In [63]:
asin = set(final_item_dataset.unique('parent_asin'))
final_review_dataset = final_review_dataset.filter(lambda x, S=asin: x['parent_asin'] in S, num_proc=10)

Flattening the indices:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter (num_proc=10): 100%|██████████| 9928825/9928825 [00:22<00:00, 441385.05 examples/s]


In [64]:
final_review_dataset

Dataset({
    features: ['rating', 'title', 'text', 'parent_asin', 'user_id'],
    num_rows: 2815510
})

In [ ]:
final_review_dataset.save_to_disk('datasets/review_dataset_final_top2000', num_proc=10)

Saving the dataset (10/10 shards): 100%|██████████| 2000/2000 [00:04<00:00, 432.45 examples/s]


Remove images links

In [81]:
def extract_first_image(x):
    # Safe extraction logic
    large_list = x['images'].get('large', [])
    x['image_url'] = large_list[0] if large_list else None
    return x

final_item_dataset = final_item_dataset.map(extract_first_image, num_proc=10)
final_item_dataset = final_item_dataset.remove_columns(['images'])

Map (num_proc=10): 100%|██████████| 2000/2000 [00:05<00:00, 342.44 examples/s]


In [82]:
final_item_dataset

Dataset({
    features: ['title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'parent_asin', 'image_url'],
    num_rows: 2000
})

In [83]:
final_item_dataset.save_to_disk('datasets/item_dataset_final_top2000', num_proc=10)

Saving the dataset (10/10 shards): 100%|██████████| 2000/2000 [00:03<00:00, 595.76 examples/s]
